In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Using a relative path from your notebook's location to the data file
file_path = "Input_Data/01_cleaned_data/uwb_dataset_02a_normalized_class_data.parquet"

# Load the parquet file into a DataFrame
df = pd.read_parquet(file_path)

# Step 2: Separate Targets (Y) from Features (X)
# We do not want to scale the 'NLOS' label or the 'RANGE' target
target_columns = ['NLOS', 'RANGE']

# Get all other columns to act as our features (FP metrics, Noise, RXPACC, and CIRs)
feature_columns = [col for col in df.columns if col not in target_columns]

X = df[feature_columns]
Y = df[target_columns]

# Step 3: Initialize the Scaler
# StandardScaler transforms data to have a mean of 0 and a standard deviation of 1.
# This is generally the best choice for PCA, SVMs, and Neural Networks.
scaler = StandardScaler()

# Step 4: Fit and Transform the features
# The scaler learns the min/max/mean of the data (fit) and applies the math (transform)
X_scaled_array = scaler.fit_transform(X)

# Step 5: Convert the scaled array back into a pandas DataFrame
# fit_transform outputs a raw numpy array, so we put the column names back on
X_scaled = pd.DataFrame(X_scaled_array, columns=feature_columns)

# Step 6: (Optional) Recombine the unscaled targets with the scaled features
# This gives you a clean, fully prepared dataset for the next steps
df_prepared = pd.concat([Y, X_scaled], axis=1)

# Display the first few rows to verify
print("Original FP_AMP1 values:", X['FP_AMP1'].head(3).values)
print("Scaled FP_AMP1 values:  ", X_scaled['FP_AMP1'].head(3).values)
print("\nPrepared Dataset Shape:", df_prepared.shape)

In [ ]:
import pandas as pd
import numpy as np

# Make a copy of the original dataframe to keep it safe
df_clean = df.copy()
print(f"Original Dataset Shape: {df_clean.shape}\n")

# ==========================================
# 1. OUTLIER REMOVAL (Using IQR Method)
# ==========================================
print("--- 1. Outlier Removal ---")
# We will check the hardware noise columns. You can add 'CIR_PWR' or 'RXPACC' to this list if needed.
columns_to_check = ['STDEV_NOISE', 'MAX_NOISE'] 

for col in columns_to_check:
    if col in df_clean.columns:
        # Calculate Q1 (25th percentile) and Q3 (75th percentile)
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        
        # Calculate Interquartile Range (IQR)
        IQR = Q3 - Q1

        # Define bounds for what is considered an "outlier" (1.5x IQR is the statistical standard)
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Count how many outliers exist before removing them
        outliers_count = len(df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)])
        print(f"Found {outliers_count} extreme outliers in '{col}'.")

        # Filter the dataframe to KEEP only rows within the normal bounds
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

print(f"Dataset Shape after Outlier Removal: {df_clean.shape}\n")


# ==========================================
# 2. DATA LEAKAGE CHECK (Using Correlation)
# ==========================================
print("--- 2. Data Leakage Check ---")
# Data leakage usually happens if a feature is basically a direct mathematical proxy for your target.
# We test this by checking the Pearson Correlation.

# To keep the calculation fast and readable, we will exclude the 1000+ raw CIR columns for this check
non_cir_cols = [c for c in df_clean.columns if not c.startswith('CIR') or c == 'CIR_PWR']

# Calculate correlation matrix for the non-CIR columns
correlation_matrix = df_clean[non_cir_cols].corr()

# Check correlations against the regression target: RANGE
if 'RANGE' in correlation_matrix.columns:
    # Get the absolute correlation values, sort them highest to lowest
    range_corr = correlation_matrix['RANGE'].abs().sort_values(ascending=False)
    
    print("Top features most correlated with the target 'RANGE':")
    # We skip the 1st one (.head(6)[1:]) because RANGE always has a 1.0 correlation with itself
    print(range_corr.head(6)[1:]) 
    
    # Check for dangerously high correlations (Greater than 95%)
    potential_leaks = range_corr[(range_corr > 0.95) & (range_corr < 1.0)]
    
    if not potential_leaks.empty:
        print("\n⚠️ WARNING: Potential Data Leakage Detected!")
        print("These features have > 0.95 correlation with RANGE. If they are distance estimates rather than raw hardware metrics, you must drop them before training:")
        print(potential_leaks)
    else:
        print("\n✅ Good news: No obvious data leakage detected (No features correlate > 0.95 with RANGE).")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

# 1. Define our columns (Assuming 'df_clean' is your dataset without outliers)
targets = ['NLOS', 'RANGE']
cir_cols = [c for c in df_clean.columns if c.startswith('CIR') and c != 'CIR_PWR']
standard_cols = [c for c in df_clean.columns if c not in targets and c not in cir_cols]

# 2. Separate Features (X) and Target (y)
X = df_clean[standard_cols + cir_cols]
y = df_clean['NLOS'] # We are classifying NLOS

# Split the data into Training and Testing sets (80% train, 20% test)
# stratify=y ensures we have a balanced number of LOS/NLOS in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ==========================================
# 3. BUILD THE PREPROCESSING PIPELINE
# ==========================================

# A. What happens to the CIR columns: Scale them, then apply PCA
cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA()) # We leave n_components blank; the Grid Search will fill it in!
])

# B. What happens to standard columns (STDEV_NOISE, FP_AMP1, etc.): Only Scale them
standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# C. Combine them using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])

# ==========================================
# 4. BUILD THE FULL MODEL PIPELINE
# ==========================================
# This connects the preprocessor directly to our proxy classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1))
])

# ==========================================
# 5. RUN THE GRID SEARCH
# ==========================================
# We tell the grid search to test PCA components from 10 to 50
param_grid = {
    'preprocessor__cir_processing__pca__n_components': [10, 15, 20, 25, 30, 35, 40, 50]
}

print("Starting Grid Search to find optimal PCA components. This may take a minute...")

# cv=5 means it will 5-fold cross-validate every single PCA number to ensure it's not a fluke
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)

# ==========================================
# 6. THE RESULTS
# ==========================================
print("\n" + "="*40)
print(f"✅ BEST PCA COMPONENTS: {grid_search.best_params_['preprocessor__cir_processing__pca__n_components']}")
print(f"✅ BEST CROSS-VAL ACCURACY: {grid_search.best_score_ * 100:.2f}%")
print("="*40 + "\n")

# If you want to see how all the other numbers performed:
results = pd.DataFrame(grid_search.cv_results_)
print("Accuracy for each number of components tested:")
for index, row in results.iterrows():
    comps = row['params']['preprocessor__cir_processing__pca__n_components']
    acc = row['mean_test_score'] * 100
    print(f" - {comps:2d} components: {acc:.2f}% accuracy")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

# Import Classifiers
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# ==========================================
# 1. LOCK IN THE OPTIMAL PREPROCESSOR
# ==========================================
OPTIMAL_PCA_COMPONENTS = 25

print(f"Building preprocessor with {OPTIMAL_PCA_COMPONENTS} CIR PCA components...\n")

cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=OPTIMAL_PCA_COMPONENTS)) 
])

standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# (Assumes cir_cols and standard_cols are still stored in memory from your previous code)
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])



In [ ]:
from sklearn.model_selection import GridSearchCV

# ==========================================
# 2. DEFINE MODELS AND THEIR HYPERPARAMETER GRIDS
# ==========================================
# Instead of just the model, we now define a grid of parameters to test for each one.
# Note: The keys in the param_grid must start with 'classifier__' so the pipeline knows 
# these settings belong to the model, not the PCA preprocessor.

models_and_params = {
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42, n_jobs=-1),
        "params": {
            'classifier__n_estimators': [50, 100],
            'classifier__max_depth': [10, 20, None]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            'classifier__max_depth': [5, 10, 15, None],
            'classifier__min_samples_split': [2, 5, 10]
        }
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            'classifier__n_neighbors': [3, 5, 7, 11],
            'classifier__weights': ['uniform', 'distance']
        }
    },
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=2000, random_state=42),
        "params": {
            'classifier__C': [0.01, 0.1, 1.0, 10.0]
        }
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {
            'classifier__var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3]
        }
    },
    "SVM (RBF Kernel)": {
        "model": SVC(kernel='rbf', probability=True, random_state=42),
        "params": {
            'classifier__C': [0.1, 1.0, 10.0],
            'classifier__gamma': ['scale', 'auto']
        }
    }
}

# Variables to store results for graphing
train_accuracies = []
test_accuracies = []
model_names = []
results_dict = {}
best_estimators = {} # We will save the perfectly tuned models here

print("Tuning, Training, and Evaluating models. This may take a few minutes...\n")

In [ ]:
# ==========================================
# STEP 3: PREPROCESSING PIPELINE
# ==========================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Lock in the optimal number discovered in the previous grid search
OPTIMAL_PCA_COMPONENTS = 25

print(f"Initializing preprocessor with {OPTIMAL_PCA_COMPONENTS} PCA components...")

# Pipeline for raw CIR Waveform data
cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=OPTIMAL_PCA_COMPONENTS)) 
])

# Pipeline for standard hardware metrics
standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Combine them: This ensures the right math is applied to the right columns
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])

print("✅ Preprocessing pipeline ready.")

In [ ]:
# ==========================================
# STEP 4: TRAINING, TUNING & VISUAL EVALUATION
# ==========================================
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import pandas as pd
import time

# (Assuming RandomForestClassifier, DecisionTreeClassifier, etc., are imported above)

# Define the dictionary of models and the parameters we want to test
models_and_params = {
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42, n_jobs=-1),
        "params": {'classifier__n_estimators': [50, 100], 'classifier__max_depth': [10, 20, None]}
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {'classifier__max_depth': [5, 10, 20, None]}
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {'classifier__n_neighbors': [3, 5, 11, 21]}
    },
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=2000, random_state=42),
        "params": {'classifier__C': [0.1, 1.0, 10.0]}
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {'classifier__var_smoothing': [1e-9, 1e-5, 1e-1]}
    },
    "SVM (RBF)": {
        "model": SVC(kernel='rbf', probability=True, random_state=42),
        "params": {'classifier__C': [0.1, 1.0, 10.0]}
    }
}

# Storage for results
final_train_accuracies = []
final_test_accuracies = []
final_model_names = []
best_estimators = {} # Fixed: Dictionary to store the best model from each GridSearch
roc_data = {}        # Storage to plot the final combined ROC curve

print("Starting Model Training and Evaluation...\n" + "="*50)

for name, config in models_and_params.items():
    print(f"Tuning and Evaluating: {name}...")
    
    # 1. Build full pipeline
    full_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', config["model"])])
    
    # 2. Setup & Fit Grid Search
    grid_search = GridSearchCV(full_pipeline, config["params"], cv=3, scoring='accuracy', n_jobs=-1, return_train_score=True)
    grid_search.fit(X_train, y_train)
    
    # 3. Extract Best Model & Make Predictions
    best_model = grid_search.best_estimator_
    best_estimators[name] = best_model
    
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1] # Probabilities for the positive/NLOS class
    
    # Save accuracies for later tabular reporting
    final_train_accuracies.append(accuracy_score(y_train, best_model.predict(X_train)) * 100)
    final_test_accuracies.append(accuracy_score(y_test, y_pred) * 100)
    final_model_names.append(name)

    # ==========================================
    # GRAPHING: 3 GRAPHS PER MODEL (1x3 Subplot)
    # ==========================================
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Comprehensive Evaluation: {name}', fontsize=16, fontweight='bold')

    # --- Graph 1: Hyperparameter Tuning ---
    cv_results = pd.DataFrame(grid_search.cv_results_)
    param_labels = [str(p).replace('classifier__', '') for p in cv_results['params']]
    
    axes[0].plot(param_labels, cv_results['mean_train_score'] * 100, label='Train Acc', marker='s', ls='--', color='skyblue')
    axes[0].plot(param_labels, cv_results['mean_test_score'] * 100, label='Test Acc (CV)', marker='o', ls='-', color='salmon')
    axes[0].set_title(f'{name}: Tuning Results')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # --- Graph 2: Confusion Matrix ---
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
    disp.plot(ax=axes[1], cmap='Blues', colorbar=False)
    axes[1].set_title(f'{name}: Confusion Matrix')

    # --- Graph 3: Individual ROC Curve ---
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    roc_data[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc} # Save data for the combined plot later
    
    axes[2].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    axes[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[2].set_xlim([0.0, 1.0])
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xlabel('False Positive Rate')
    axes[2].set_ylabel('True Positive Rate')
    axes[2].set_title(f'{name}: ROC Curve')
    axes[2].legend(loc="lower right")

    # Display the 1x3 dashboard for this specific model
    plt.tight_layout()
    plt.subplots_adjust(top=0.85) # Give the suptitle breathing room
    plt.show()

print("\n" + "="*50 + "\nGenerating Final Combined ROC Curve...")

# ==========================================
# STEP 5: COMBINED ROC CURVE SUMMARY
# ==========================================
fig_roc, ax_roc = plt.subplots(figsize=(10, 8))

# Plot all saved ROC data onto one graph
for model_name, data in roc_data.items():
    ax_roc.plot(data['fpr'], data['tpr'], lw=2, label=f'{model_name} (AUC = {data["auc"]:.3f})')

# Formatting the combined graph
ax_roc.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess (AUC=0.500)')
ax_roc.set_xlim([0.0, 1.0])
ax_roc.set_ylim([0.0, 1.05])
ax_roc.set_xlabel('False Positive Rate (Fall-out)', fontsize=12, fontweight='bold')
ax_roc.set_ylabel('True Positive Rate (Recall)', fontsize=12, fontweight='bold')
ax_roc.set_title('Combined ROC Curve: All Models', fontsize=16, fontweight='bold')
ax_roc.legend(loc="lower right", fontsize=11)
ax_roc.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# ==========================================
# STEP 5: FINAL COMPARISON SUMMARY
# ==========================================
import numpy as np

x = np.arange(len(final_model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
rects1 = ax.bar(x - width/2, final_train_accuracies, width, label='Best Train Accuracy', color='skyblue', edgecolor='black')
rects2 = ax.bar(x + width/2, final_test_accuracies, width, label='Best Test Accuracy', color='salmon', edgecolor='black')

ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('Final Comparison: Best Tuned Models (Train vs. Test)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(final_model_names, rotation=45, ha='right')
ax.set_ylim(0, 115)
ax.legend(loc='lower right')
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Labeling bars
for rect in rects1 + rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()